# Project 1

Welcome to the first NLP1 project. The goal of the first tutorial was to get you familiar with text segmentation and simple text classification for topic detection.
Given the skills you have developed in the first week, you will now be applying them within a different dataset and task.

We are recommending a set of tasks and associated datasets; using those, you will be experimenting on problems introduced during the tutorials. You will be applying hands-on the experiments you have already explored in the last tutorial, and taking a deeper dive: extending your experiments, trying different set ups, possibly considering different evaluation techniques and analysing errors. 

The week after the deadline, your TA will perform a short oral assessment of the skills and knowledge you acquired (make sure to check the official schedule for this), probing you to answer questions about what you learnt, about what you did and about why you did the things you did (as opposed to some reasonable alternatives) as well as pros and cons of your method, and so on. 

As the tutorials will be building upon your knowledge throughout the weeks, the task and dataset you choose now will be the ones you use in projects 2 and 3; and the ones you will be finally writing about in your final blog post.  

## Get to work!

The next section has a checklist to help you not lose track of what needs to be done. Here we have a more informal guide, with prompts and questions for you. 


1. [Choose a dataset](http://nlp-illc.github.io/nlp1/). Spend some time understanding how the dataset looks and some statistics of it (e.g. how many instances of each classification label we have?)
2. Remember what we learnt during the tutorial: to develop any classifier of the kind we saw, we needed a way to segment text. Do you want to use the tokeniser we have developed? Do you need to train a new one? If you do, what hyperparameters shall you use? What data shall you use to train it? Consider that your dataset may be too small to train a tokeniser! Think whether you will need a larger resource of similar nature to train a tokeniser.
3. Start thinking how to represent your documents into features. We have experimented with token occurences and frequencies. Can you think of any other ways, possibly? 
4. We had experimented with Logistic Regression and a shallow neural network. You can experiment with and assess different architectures now and their hyper parameters (e.g. add layers, possibly change the activation functions, drop out rates).
5. Evaluate your models! Was the evaluation method adequate for your new experimental setup? Do you want to assess other aspects of performance?
6. Analyse the various models you assessed and try to decide on which is the best.

Be mindful that you will be using your experiments to write up your final blog, so make sure they are clean and neat!

You can write your own code and you can reuse code from the tutorial and its packages or from other sources. You may use coding assistants if you like, but use it mindfully as you take full responsibility for this kind of (semi-)automated coding. Also, keep a record of the kinds of things you delegate to any form of assistant, as you will be asked to disclose this in your blog post (not because we will pass a value judgement on it, but because it is a widespread and good practice to document such things).

You may use jupyter notebooks or python files, it's up to you as we will not directly assess your code. But, once again, be organised, as we will ask you to share a reproducible version of your experiments along with your blog post (again, because this is how real research is publicised). 

If you are unsure about anything, ask your TA! They are always happy to help you! Happy coding! :)

## Checklist

What **we expect you to do**:

- [ ] Compare different systems of text classification for your choice of data/task.
- Your comparisons should cover:
    - [ ] the two representation spaces in the tutorial (token indicators and bag-of-words) and an additional one of your choice.
    - [ ] the two kinds of architectures in the tutorial (log-linear and single-hidden-layer MLP) and an additional one of your choice (such as a modification of the MLP).
- [ ] Your comparisons must be based on cross-validation and a relevant metric (accuracy or else).
- [ ] You should make use of Bayesian estimation to help you analyse results in relation to a region of practical equivalence (ROPE), and you can motivate the ROPE size yourself.

What you need to **bring to the oral assessment**:
* You will be asked questions about the points above, you can bring a slide deck with up to five slides (one for each of the following topics _data and metrics_, _features_, _architectures_, _results_, _extra_); you _will not_ give a presentation, rather, these slides are to be thought of as "backup slides" that you can use to guide your answers to the questions the TA will ask you.
* You may also be asked questions about the tutorial, for those, you don't need to prepare any deliverable (a good way to prepare is to work through the tutorial and reflect on the questions there).

What you should **keep in mind** given that you will eventually write a blog post about your work throughout the course:
* Organise your code;
* Summarise your motivations and findings;
* Document your sources and any form of AI-assistance you use.

In [12]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import part1
from datasets import load_dataset

dataset = load_dataset("google-research-datasets/paws", "labeled_final")
paws_outcome_space = part1.clslib.NamedClasses(['not paraphrase', 'paraphrase'])


## WE use the PAWS dataset
paws consist of a dataset of two sentences with a binary classsification with the question if two of the sentences are a pharaphrase of the other.


# Dataset statistics and examples

In [4]:
#An example of a sentence pair from our training corpus
print("Sentence 1:", dataset["train"]["sentence1"][0])
print("Sentence 2:", dataset["train"]["sentence2"][0])
print("Gold label (id):", dataset["train"]["label"][0])

Sentence 1: In Paris , in October 1560 , he secretly met the English ambassador , Nicolas Throckmorton , asking him for a passport to return to England through Scotland .
Sentence 2: In October 1560 , he secretly met with the English ambassador , Nicolas Throckmorton , in Paris , and asked him for a passport to return to Scotland through England .
Gold label (id): 0


In [15]:
from collections import Counter
from statistics import mean, median

def count_labels(data):
    """
    count the number of data points for weach of the binary labels in the dataset
    """
    label_counts = Counter(data["label"])
    return label_counts

def print_precentages(label_counts, total):
    """
    print the percentage of each label in the dataset
    """
    for label, count in label_counts.items():
        percentage = (count / total) * 100
        print(f"Label {label}: {percentage }%")

def entence_length_stats(data):
    """
    compute the mean and median length of the sentences in the dataset
    """
    sentence_lengths = [(len(s1.split()), len(s2.split()), label) for s1, s2, label in zip(data["sentence1"], data["sentence2"], data["label"])]
    mean_length_s1 = mean(length[0] for length in sentence_lengths)
    mean_length_s2 = mean(length[1] for length in sentence_lengths)
    median_length_s1 = median(length[0] for length in sentence_lengths)
    median_length_s2 = median(length[1] for length in sentence_lengths)
    print(f"Mean length of sentence 1 (based on spaces): {mean_length_s1}")
    print(f"Mean length of sentence 2 (based on spaces): {mean_length_s2}")
    print(f"Median length of sentence 1 (based on spaces): {median_length_s1}")
    print(f"Median length of sentence 2 (based on spaces): {median_length_s2}")
    return mean_length_s1, mean_length_s2, median_length_s1, median_length_s2

print("Training set label counts:")
train_label_counts = count_labels(dataset["train"])
print_precentages(train_label_counts, len(dataset["train"]))
_ = entence_length_stats(dataset["train"])

Training set label counts:
Label 0: 55.81263537175361%
Label 1: 44.18736462824639%
Mean length of sentence 1 (based on spaces): 21.340782575251513
Mean length of sentence 2 (based on spaces): 21.337948624521772
Median length of sentence 1 (based on spaces): 21
Median length of sentence 2 (based on spaces): 21


In [5]:
import part1

In [11]:
# PAWS has no "text" column: use both sentences of every pair as the corpus
corpus = list(dataset["train"]["sentence1"]) 
print("datapoints: ", len(corpus))
print()


datapoints:  49401


In [7]:
vocab_size = 5000
min_frequency = 2
tokenizer = part1.experiment.train_tokenizer(corpus, vocab_size, min_frequency)

# tokenize the data
1) Token indicators (like the tutorial)
2) Bag of words (home made)
3) UNKOWN

In [ ]:
#Token indicator code
import numpy as np  
from tokenizers import Tokenizer
class Encoder:

    def __len__(self) -> int:
        """Dimensionality of the representation space"""
        raise NotImplementedError("Concrete implementations will be discussed later")

    def __call__(self, x: str) -> 'np.array':
        """Represent x as a D-dimensional vector of real numbers"""
        raise NotImplementedError("Concrete implementations will be discussed later")
        
class TokenIndicator(Encoder):

    def __init__(self, tokenizer: Tokenizer):
        self.tokenizer = tokenizer
        
    def __len__(self) -> int:
        """Dimensionality of the representation space"""
        return len(tokenizer.get_vocab())

    def __call__(self, x: str) -> 'np.array':
        """Represent x as a D-dimensional vector of real numbers"""
        unique_token_ids = list(set(self.tokenizer.encode(x).ids))

        # A 0 vector with the same length as the vocabulary except in positions corresponding 
        # to tokens appearing in the document
        vector = np.zeros(len(self))
        vector[unique_token_ids] = 1
        return vector

data_token_indicator = TokenIndicator(tokenizer)


In [29]:
# bag of words representation
class BagOfWords(Encoder):
    def __init__(self, tokenizer: Tokenizer):
        self.tokenizer = tokenizer

    def __len__(self) -> int:
        """Dimensionality of the representation space"""
        return len(self.tokenizer.get_vocab())

    def __call__(self, x: str) -> 'np.array':
        """Represent x as a D-dimensional vector of real numbers"""
        encoded = self.tokenizer.encode(x)
        vector = np.zeros(len(self))
        for token_id in encoded.ids:
            vector[token_id] += 1
        return vector

data_bag_of_words = BagOfWords(tokenizer)

In [33]:
#example of both tkenizers
print("Sentence 1:", dataset["train"]["sentence1"][0])
print("Token Indicator representation:", data_token_indicator(dataset["train"]["sentence1"][0]))
print("total number of tokens in the sentence:", sum(data_token_indicator(dataset["train"]["sentence1"][0])))
#example of the representation of a sentence using the BagOfWords
print("Sentence 1:", dataset["train"]["sentence1"][0])
print("Bag of Words representation:", data_bag_of_words(dataset["train"]["sentence1"][0]))
print("total number of tokens in the sentence:", sum(data_bag_of_words(dataset["train"]["sentence1"][0])))

Sentence 1: In Paris , in October 1560 , he secretly met the English ambassador , Nicolas Throckmorton , asking him for a passport to return to England through Scotland .
Token Indicator representation: [0. 0. 0. ... 0. 0. 0.]
total number of tokens in the sentence: 37.0
Sentence 1: In Paris , in October 1560 , he secretly met the English ambassador , Nicolas Throckmorton , asking him for a passport to return to England through Scotland .
Bag of Words representation: [0. 0. 0. ... 0. 0. 0.]
total number of tokens in the sentence: 41.0


# Train calssiefieres
